# GLP-1 Drug-Firm Matching and Sentiment Index

This notebook assigns documents to drugs and pharmaceutical firms using rule-based matching, then constructs a sentiment index for the top 10 firms in the GLP-1 market.

**Objective**: 
- Match articles to specific GLP-1 drugs and manufacturers
- Build sentiment indicators from text features
- Analyze market sentiment across pharmaceutical firms

## Section 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("✓ Libraries imported successfully")

## Section 2: Define Drug and Firm Mapping Dictionaries

Create comprehensive mappings of:
- **Drug Map**: Each canonical drug name with brand names, aliases, and related product names
- **Firm Map**: Which firm manufactures each drug

In [ ]:
# Drug mapping: canonical name -> brand names and aliases
drug_map = {
    "semaglutide": ["semaglutide", "ozempic", "wegovy", "rybelsus"],
    "tirzepatide": ["tirzepatide", "mounjaro", "zepbound"],
    "liraglutide": ["liraglutide", "saxenda", "victoza"],
    "dulaglutide": ["dulaglutide", "trulicity"],
    "danuglipron": ["danuglipron"],
    "maritide": ["maritide", "amg133"],
    "vk2735": ["vk2735"],
    "ct388": ["ct388", "carmot"],
    "ct996": ["ct996"],
    "glp1_gip": ["glp-1/gip", "glp1/gip", "dual-acting"]
}

# Firm mapping: drug -> manufacturer (top 10 firms)
firm_map = {
    "semaglutide": "Novo Nordisk A/S",
    "liraglutide": "Novo Nordisk A/S",
    "dulaglutide": "Eli Lilly and Company",
    "tirzepatide": "Eli Lilly and Company",
    "maritide": "Amgen Inc.",
    "danuglipron": "Pfizer Inc.",
    "ct388": "Roche Holding AG",
    "ct996": "Roche Holding AG",
    "glp1_gip": "AstraZeneca Plc",
    "vk2735": "Viking Therapeutics, Inc.",
}

# Top 10 firms with description and investment thesis
top_10_firms = {
    "Novo Nordisk A/S": {
        "description": "Commands the market with blockbuster drugs semaglutide (Ozempic for diabetes, Wegovy for obesity) and Rybelsus (oral)",
        "drugs": ["semaglutide", "liraglutide"],
        "strength": "Market leader with approved obesity drug (Wegovy)",
    },
    "Eli Lilly and Company": {
        "description": "Holds commanding market share with tirzepatide (Mounjaro for diabetes, Zepbound for obesity) and is rapidly expanding portfolio",
        "drugs": ["tirzepatide", "dulaglutide"],
        "strength": "Fast-growing competitor with strong obesity portfolio",
    },
    "Amgen Inc.": {
        "description": "Developing MariTide (formerly AMG133), highly anticipated antibody-based GLP-1/GIP therapy in Phase 3 trials",
        "drugs": ["maritide"],
        "strength": "Novel next-generation candidate",
    },
    "Pfizer Inc.": {
        "description": "Developing next-generation oral GLP-1 receptor agonists (danuglipron) and has significant presence in metabolic R&D",
        "drugs": ["danuglipron"],
        "strength": "Oral formulation innovation",
    },
    "Roche Holding AG": {
        "description": "Entered GLP-1 race aggressively through $2.7B acquisition of Carmot Therapeutics with clinical-stage injectable (CT-388) and oral (CT-996)",
        "drugs": ["ct388", "ct996"],
        "strength": "Aggressive M&A strategy, both injectable and oral",
    },
    "AstraZeneca Plc": {
        "description": "Long-standing player continuing to advance portfolio with small-molecule oral GLP-1 candidates",
        "drugs": ["glp1_gip"],
        "strength": "Cardiovascular and metabolic expertise",
    },
    "Zealand Pharma A/S": {
        "description": "Developer of peptide-based medicines with strong pipeline for gastrointestinal and metabolic disorders",
        "drugs": [],
        "strength": "Acquisition target potential",
    },
    "Structure Therapeutics, Inc.": {
        "description": "Leads development of oral, small-molecule GLP-1 receptor agonists designed to replace injections",
        "drugs": [],
        "strength": "Oral formulation specialization",
    },
    "Viking Therapeutics, Inc.": {
        "description": "Known for promising dual GLP-1/GIP agonist VK2735 showing positive weight loss results in subcutaneous and oral formulations",
        "drugs": ["vk2735"],
        "strength": "Dual-mechanism approach with positive trial data",
    },
    "Boehringer Ingelheim GmbH": {
        "description": "Combines cardiovascular expertise with metabolic research, focusing on dual-acting GLP-1 candidates",
        "drugs": [],
        "strength": "Cardiometabolic integration",
    },
}

print(f"✓ Drug mapping: {len(drug_map)} drugs defined")
print(f"✓ Firm mapping: {len(firm_map)} drug-firm associations")
print(f"✓ Top 10 firms configured")

## Section 3: Create Drug Tagging Function

Define regex-based function to identify drug mentions in text.
- Case-insensitive matching
- Word boundary detection (prevents "ozempic" matching within other words)
- Handles multiple drugs per document

In [ ]:
def tag_drug(text: str) -> list:
    """
    Identify all drug mentions in text using regex with word boundaries
    
    Args:
        text: Text to search for drug mentions
        
    Returns:
        List of canonical drug names found (deduplicated)
    """
    if pd.isna(text):
        return []
    
    text = str(text).lower()
    found = []
    
    # Check each canonical drug and its keywords
    for drug, keywords in drug_map.items():
        for keyword in keywords:
            # Use word boundary \b to prevent partial matches
            # e.g., "ozempic" won't match "ozempic123" or "myozempic"
            if re.search(rf"\b{re.escape(keyword)}\b", text):
                found.append(drug)
                break  # Only add drug once per document
    
    # Return deduplicated list
    return list(set(found))


# Test the function
test_texts = [
    "Ozempic and Wegovy are semaglutide drugs from Novo Nordisk",
    "Mounjaro tirzepatide shows promise for weight loss",
    "No drug mentions here",
    "Both Ozempic (semaglutide) and Mounjaro (tirzepatide) are GLP-1 drugs"
]

print("Testing drug tagging function:\n")
for text in test_texts:
    tags = tag_drug(text)
    print(f"Text: {text[:60]}...")
    print(f"Drugs found: {tags}\n")

## Load or Create Sample Data

Load data from the ingestion pipeline or create sample data for demonstration

In [ ]:
# Try to load from data_ingestion pipeline, else create sample data
try:
    from data_ingestion import fetch_glp1_data
    print("Loading data from ingestion pipeline...")
    df = fetch_glp1_data(fetch_news=True, fetch_reddit=True)
    
    if len(df) == 0:
        raise ValueError("No data from pipeline")
    print(f"✓ Loaded {len(df)} documents from pipeline\n")
    
except Exception as e:
    print(f"Pipeline not available ({e}), using sample data\n")
    
    # Create sample dataset for demonstration
    sample_data = {
        "text": [
            "Novo Nordisk's Ozempic shows strong performance in diabetes market. Wegovy obesity treatment gaining traction.",
            "Eli Lilly tirzepatide (Mounjaro) outperforms expectations in clinical trials with rapid weight loss results.",
            "Amgen's MariTide demonstrates promising Phase 3 results for dual GLP-1/GIP mechanism.",
            "Pfizer advances danuglipron oral GLP-1 agonist with next-generation formulation strategy.",
            "Roche acquisition of Carmot Therapeutics adds CT-388 injectable and CT-996 oral candidates to pipeline.",
            "AstraZeneca focuses on small-molecule oral GLP-1 candidates combining cardiovascular benefits.",
            "Viking Therapeutics VK2735 shows positive weight loss data superior to current competitors.",
            "Structure Therapeutics develops oral GLP-1 agonists designed to replace injection formulations.",
            "No drug mention in this FDA regulatory update about approval timelines.",
            "Novo Nordisk Rybelsus oral semaglutide gaining momentum in weight loss segment.",
            "Dulaglutide (Trulicity) by Eli Lilly expands indication coverage amid GLP-1 class growth.",
            "Boehringer Ingelheim cardiovascular expertise combined with GLP-1 development strategy.",
            "Zealand Pharma acquisition potential increases as GLP-1 consolidation accelerates.",
            "Top pharma companies racing to develop improved GLP-1 formulations and delivery methods.",
            "Saxenda liraglutide demand increases alongside Novo Nordisk's blockbuster portfolio expansion.",
        ],
        "title": [
            "Novo Nordisk leads GLP-1 market with Ozempic and Wegovy",
            "Eli Lilly's Mounjaro tirzepatide shows clinical superiority",
            "Amgen MariTide Phase 3 results exceed expectations",
            "Pfizer danuglipron oral formulation strategy",
            "Roche $2.7B Carmot acquisition strengthens GLP-1 position",
            "AstraZeneca small-molecule GLP-1 program advances",
            "Viking VK2735 demonstrates weight loss leadership",
            "Structure Therapeutics oral innovation pipeline",
            "FDA regulatory update",
            "Novo Nordisk Rybelsus momentum builds",
            "Eli Lilly Trulicity expansion amid GLP-1 competition",
            "Boehringer Ingelheim metabolic therapy focus",
            "Zealand Pharma M&A speculation intensifies",
            "Pharma GLP-1 innovation race accelerates",
            "Novo Nordisk Saxenda demand surge",
        ],
        "source": ["NewsAPI"] * 15,
        "timestamp": pd.date_range(start="2026-03-01", periods=15, freq="D"),
        "url": [f"https://example.com/article{i}" for i in range(15)],
        "author": ["Reporter"] * 15,
    }
    
    df = pd.DataFrame(sample_data)
    print(f"✓ Created sample dataset with {len(df)} documents\n")

print(f"Dataset shape: {df.shape}")
print(f"\nFirst 3 documents:")
print(df[["title", "source", "timestamp"]].head(3))

## Section 4: Apply Drug Tags to Documents

Apply the tagging function to identify all drug mentions in each document

In [ ]:
print("Applying drug tags to documents...")
df["drugs"] = df["text"].apply(tag_drug)
print(f"✓ Tagged {len(df)} documents\n")

# Show tagging results
print("Sample tagging results:")
print("─" * 80)
for idx, row in df[["title", "drugs"]].head(10).iterrows():
    drugs_str = ", ".join(row["drugs"]) if row["drugs"] else "[no drugs found]"
    print(f"{row['title'][:50]:50s} → {drugs_str}")

print("\n" + "─" * 80)
print(f"\nTagging statistics:")
print(f"  Documents with drugs: {(df['drugs'].str.len() > 0).sum()}")
print(f"  Documents without drugs: {(df['drugs'].str.len() == 0).sum()}")
print(f"  Total drug mentions: {df['drugs'].apply(len).sum()}")

# Show drug distribution
print(f"\nDrug mention frequency:")
drug_counts = Counter()
for drug_list in df["drugs"]:
    drug_counts.update(drug_list)

for drug, count in drug_counts.most_common():
    print(f"  {drug:20s}: {count:3d} mentions")

## Section 5: Expand Rows for Multi-Drug Mentions

Some documents mention multiple drugs. Expand rows so each drug gets its own row.
This allows sentiment analysis per drug while preserving document context.

In [ ]:
print("Expanding rows for multi-drug documents...")
df_original_count = len(df)

# Explode the drugs column so each drug gets its own row
df_expanded = df.explode("drugs").reset_index(drop=True)

print(f"✓ Expanded from {df_original_count} documents to {len(df_expanded)} drug-document pairs\n")

# Show example of expansion
print("Example: Document with multiple drugs:")
print("─" * 80)
multi_drug_idx = [i for i, d in enumerate(df["drugs"]) if len(d) > 1][0]
print(f"\nOriginal document #{multi_drug_idx}:")
print(f"  Title: {df.iloc[multi_drug_idx]['title']}")
print(f"  Drugs: {df.iloc[multi_drug_idx]['drugs']}")

print(f"\nAfter expansion:")
expanded_rows = df_expanded[df_expanded["text"] == df.iloc[multi_drug_idx]["text"]]
for idx, row in expanded_rows.iterrows():
    print(f"  Row {idx}: {row['drugs']}")

# Show distribution
print(f"\n" + "─" * 80)
print(f"\nExpansion statistics:")
print(f"  Original documents: {df_original_count}")
print(f"  Expanded rows: {len(df_expanded)}")
print(f"  Expansion factor: {len(df_expanded) / df_original_count:.2f}x")

docs_with_drugs = (df["drugs"].str.len() > 0).sum()
docs_with_multi_drugs = (df["drugs"].str.len() > 1).sum()
print(f"  Documents with 1 drug: {docs_with_drugs - docs_with_multi_drugs}")
print(f"  Documents with 2+ drugs: {docs_with_multi_drugs}")

## Section 6: Map Drugs to Firms

Link each drug to its manufacturer using the firm_map, then filter to top 10 firms

In [ ]:
print("Mapping drugs to firms...")

# Map drugs to firms
df_expanded["firm"] = df_expanded["drugs"].map(firm_map)

# Count by firm
firm_counts = df_expanded["firm"].value_counts()
print(f"\n✓ Drug-to-firm mapping complete\n")
print(f"Firm representation in expanded data:")
print("─" * 40)
for firm, count in firm_counts.items():
    print(f"  {firm:30s}: {count:3d} rows")

# Filter to top 10 firms only
top_10_firm_names = list(top_10_firms.keys())
df_filtered = df_expanded[df_expanded["firm"].isin(top_10_firm_names)].copy()

print(f"\n" + "─" * 40)
print(f"After filtering to top 10 firms:")
print(f"  Original expanded rows: {len(df_expanded)}")
print(f"  Rows with firm mapping: {(df_expanded['firm'].notna()).sum()}")
print(f"  Rows in top 10 firms: {len(df_filtered)}")
print(f"  Retention rate: {len(df_filtered) / len(df_expanded) * 100:.1f}%")

# Show example rows
print(f"\n" + "─" * 40)
print("Sample matched rows:")
print(df_filtered[["title", "drugs", "firm"]].head(10).to_string(index=False))

## Section 7: Clean Data and Remove Irrelevant Rows

Remove rows where:
- No drug was identified (firms = NaN)
- Drug is not in top 10 firm portfolio
- Text is too short or missing

In [ ]:
print("Cleaning data...")

# Create working copy
df_clean = df_filtered.copy()

# Remove rows with NaN firm (no mapping found)
rows_before = len(df_clean)
df_clean = df_clean.dropna(subset=["firm"])
rows_removed_no_firm = rows_before - len(df_clean)

# Remove rows with empty or very short text
rows_before = len(df_clean)
df_clean = df_clean[df_clean["text"].notna()]
df_clean = df_clean[df_clean["text"].str.len() > 20]
rows_removed_short = rows_before - len(df_clean)

# Reset index
df_clean = df_clean.reset_index(drop=True)

print(f"✓ Data cleaned\n")
print(f"Cleaning statistics:")
print(f"  Rows removed (no firm mapping): {rows_removed_no_firm}")
print(f"  Rows removed (empty/short text): {rows_removed_short}")
print(f"  Final clean dataset: {len(df_clean)} rows")

# Show firm distribution in clean data
print(f"\n" + "─" * 60)
print("Distribution across top 10 firms:")
firm_dist = df_clean["firm"].value_counts()
for firm, count in firm_dist.items():
    bar = "█" * (count // 2 + 1) if count > 0 else ""
    pct = count / len(df_clean) * 100
    print(f"  {firm:30s}: {count:3d} ({pct:5.1f}%) {bar}")

# Add more useful columns
print(f"\n" + "─" * 60)
print("Adding derived features...")

df_clean["text_length"] = df_clean["text"].str.len()
df_clean["word_count"] = df_clean["text"].str.split().str.len()
df_clean["has_url"] = df_clean["url"].notna()

# Add drug strength indicator (for more recent trials, more mentions)
df_clean["drug_strength"] = df_clean["drugs"].apply(
    lambda x: "core" if x in ["semaglutide", "tirzepatide"] else "pipeline"
)

print(f"✓ Features added: text_length, word_count, has_url, drug_strength")
print(f"\nFinal clean dataset shape: {df_clean.shape}")
print("\nFirst 5 rows of clean data:")
print(df_clean[["title", "drugs", "firm", "timestamp", "text_length"]].head().to_string(index=False))

## Section 8: Top 10 Firms Portfolio Overview

Display portfolio composition and investment thesis for each firm

In [ ]:
# Create firms reference dataframe
firms_data = []
for rank, (firm_name, info) in enumerate(top_10_firms.items(), 1):
    article_count = len(df_clean[df_clean["firm"] == firm_name])
    drugs_in_data = [d for d in info["drugs"] if d in df_clean["drugs"].unique()]
    
    firms_data.append({
        "Rank": rank,
        "Firm": firm_name,
        "Description": info["description"],
        "Strength": info["strength"],
        "Articles": article_count,
        "Drugs in Data": ", ".join(drugs_in_data) if drugs_in_data else "—",
    })

df_firms = pd.DataFrame(firms_data)

print("TOP 10 PHARMACEUTICAL FIRMS - GLP-1 MARKET")
print("=" * 120)
print(df_firms[["Rank", "Firm", "Strength", "Articles", "Drugs in Data"]].to_string(index=False))

print("\n" + "=" * 120)
print("\nDETAILED FIRM PROFILES:")
for rank, firm_name in enumerate(top_10_firms.keys(), 1):
    info = top_10_firms[firm_name]
    article_count = len(df_clean[df_clean["firm"] == firm_name])
    
    print(f"\n{rank}. {firm_name}")
    print("   " + "─" * 100)
    print(f"   Description: {info['description']}")
    print(f"   Competitive Strength: {info['strength']}")
    print(f"   Articles in data: {article_count}")
    if info["drugs"]:
        print(f"   Key Drugs: {', '.join(info['drugs'])}")

print("\n" + "=" * 120)

## Section 9: Extract Sentiment Features

Calculate sentiment indicators from text:
- Simple positive/negative keyword counts
- Sentiment tone indicators (trial results, clinical data)
- Article relevance and quality metrics

In [ ]:
print("Extracting sentiment features...")

# Define sentiment keywords
positive_keywords = [
    "positive", "promising", "success", "breakthrough", "approved", "efficacy",
    "superior", "outperform", "excellence", "innovation", "leading",
    "strong", "powerful", "effective", "impressive", "remarkable",
    "gain", "momentum", "surge", "expansion", "growth", "advance", "progress"
]

negative_keywords = [
    "negative", "declined", "failed", "setback", "adverse", "concern",
    "risk", "delay", "loss", "weak", "inferior", "criticism", "competition",
    "challenge", "pressure", "loss share", "decline", "reduction"
]

trial_keywords = [
    "trial", "clinical", "data", "phase", "fda", "approval", "approved",
    "efficacy", "safety", "efficacy", "endpoints", "results", "significant"
]

def extract_sentiment_features(text):
    \"\"\"Extract sentiment indicators from text\"\"\"
    text_lower = str(text).lower()
    
    return {
        "positive_count": sum(1 for kw in positive_keywords if kw in text_lower),
        "negative_count": sum(1 for kw in negative_keywords if kw in text_lower),
        "trial_count": sum(1 for kw in trial_keywords if kw in text_lower),
        "has_clinical_data": any(kw in text_lower for kw in trial_keywords),
    }

# Apply feature extraction
sentiment_features = df_clean["text"].apply(extract_sentiment_features)
df_clean = df_clean.join(pd.DataFrame(sentiment_features.tolist()))

# Create composite sentiment score
df_clean["sentiment_net"] = df_clean["positive_count"] - df_clean["negative_count"]
df_clean["sentiment_ratio"] = (df_clean["positive_count"] + 1) / (df_clean["negative_count"] + 1)
df_clean["content_quality"] = df_clean["trial_count"] * 2 + df_clean["word_count"] / 100

print(f"✓ Sentiment features extracted")
print(f"\nFeature statistics:")
print(f"  Positive keywords - Mean: {df_clean['positive_count'].mean():.2f}, Max: {df_clean['positive_count'].max()}")
print(f"  Negative keywords - Mean: {df_clean['negative_count'].mean():.2f}, Max: {df_clean['negative_count'].max()}")
print(f"  Trial/Clinical refs - Mean: {df_clean['trial_count'].mean():.2f}, Max: {df_clean['trial_count'].max()}")
print(f"  Articles with clinical data: {df_clean['has_clinical_data'].sum()}")

# Show examples
print(f"\n" + "─" * 100)
print("Sample sentiment features:")
print(df_clean[["title", "positive_count", "negative_count", "sentiment_net", "has_clinical_data"]].head(10).to_string(index=False))

## Section 10: Calculate Sentiment Index

Aggregate sentiment features at firm level to create sentiment indices:
- **Article Count**: Number of articles mentioning the firm
- **Average Sentiment**: Mean of net sentiment scores
- **Positive Ratio**: Proportion of positive articles
- **Clinical Evidence**: Articles with trial/clinical data
- **Content Quality Index**: Weighted index based on article depth

In [ ]:
print("Calculating firm-level sentiment indices...")

# Group by firm and aggregate
sentiment_index = df_clean.groupby("firm").agg({
    "text": "count",  # Article count
    "sentiment_net": ["mean", "std", "min", "max"],
    "sentiment_ratio": "mean",
    "positive_count": "mean",
    "negative_count": "mean",
    "has_clinical_data": ["sum", "mean"],
    "content_quality": "mean",
    "word_count": "mean",
}).round(3)

# Flatten column names
sentiment_index.columns = ["_".join(col).strip("_") for col in sentiment_index.columns.values]
sentiment_index = sentiment_index.rename(columns={"text_count": "article_count"})

# Calculate additional metrics
sentiment_index["positive_ratio"] = (df_clean.groupby("firm")["positive_count"] > 0).sum() / sentiment_index["article_count"]
sentiment_index["clinical_articles"] = df_clean.groupby("firm")["has_clinical_data"].sum()
sentiment_index["clinical_ratio"] = sentiment_index["clinical_articles"] / sentiment_index["article_count"]

# Create composite sentiment index (normalized)
sentiment_index["sentiment_index"] = (
    sentiment_index["sentiment_net_mean"] * 0.4 +
    sentiment_index["positive_ratio"] * 100 * 0.3 +
    sentiment_index["clinical_ratio"] * 100 * 0.3
).round(2)

# Sort by sentiment index
sentiment_index = sentiment_index.sort_values("sentiment_index", ascending=False)

print(f"✓ Sentiment index calculated for {len(sentiment_index)} firms")

# Display results
print(f"\n" + "=" * 120)
print("SENTIMENT INDEX BY FIRM (Sorted by Index)")
print("=" * 120)

display_df = sentiment_index[
    ["article_count", "sentiment_net_mean", "positive_ratio", "clinical_ratio", "sentiment_index"]
].copy()
display_df.columns = ["Articles", "Avg Sentiment", "Positive Ratio", "Clinical Ratio", "Sentiment Index"]
display_df["Articles"] = display_df["Articles"].astype(int)
display_df["Positive Ratio"] = (display_df["Positive Ratio"] * 100).round(1).astype(str) + "%"
display_df["Clinical Ratio"] = (display_df["Clinical Ratio"] * 100).round(1).astype(str) + "%"

print(display_df.to_string())

# Detailed breakdown
print(f"\n" + "=" * 120)
print("DETAILED SENTIMENT ANALYSIS BY FIRM:")
print("=" * 120)

for rank, (firm, row) in enumerate(sentiment_index.iterrows(), 1):
    print(f"\n{rank}. {firm}")
    print("   " + "─" * 110)
    print(f"   Articles: {int(row['article_count'])}")
    print(f"   Avg Sentiment Score: {row['sentiment_net_mean']:.2f} (range: {row['sentiment_net_min']:.0f} to {row['sentiment_net_max']:.0f})")
    print(f"   Positive Tone Ratio: {row['positive_ratio']*100:.1f}%")
    print(f"   Clinical Data Articles: {int(row['clinical_articles'])} / {int(row['article_count'])} ({row['clinical_ratio']*100:.1f}%)")
    print(f"   Content Quality Index: {row['content_quality']:.2f}")
    print(f"   📊 SENTIMENT INDEX: {row['sentiment_index']:.1f}")

## Section 11: Visualize Results

Create visualizations of sentiment indices across firms

In [ ]:
# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 10)

# Create a 2x2 subplot layout
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("GLP-1 Market Sentiment Index - Top 10 Firms", fontsize=16, fontweight="bold")

# 1. Sentiment Index by Firm (Bar chart)
ax = axes[0, 0]
sentiment_index_sorted = sentiment_index.sort_values("sentiment_index", ascending=True)
colors = ["#2ecc71" if x > 0 else "#e74c3c" for x in sentiment_index_sorted["sentiment_index"]]
sentiment_index_sorted["sentiment_index"].plot(kind="barh", ax=ax, color=colors, edgecolor="black")
ax.set_xlabel("Sentiment Index", fontweight="bold")
ax.set_title("Overall Sentiment Index by Firm", fontweight="bold")
ax.axvline(x=0, color="black", linestyle="-", linewidth=0.8)
ax.grid(axis="x", alpha=0.3)

# 2. Article Count by Firm (Bar chart)
ax = axes[0, 1]
sentiment_index_sorted["article_count"].plot(kind="barh", ax=ax, color="#3498db", edgecolor="black")
ax.set_xlabel("Number of Articles", fontweight="bold")
ax.set_title("Article Mentions by Firm", fontweight="bold")
ax.grid(axis="x", alpha=0.3)

# 3. Sentiment vs Article Count (Scatter plot)
ax = axes[1, 0]
scatter = ax.scatter(
    sentiment_index["article_count"],
    sentiment_index["sentiment_index"],
    s=300,
    alpha=0.6,
    c=sentiment_index["clinical_ratio"],
    cmap="RdYlGn",
    edgecolors="black",
    linewidth=2
)
ax.set_xlabel("Article Count", fontweight="bold")
ax.set_ylabel("Sentiment Index", fontweight="bold")
ax.set_title("Sentiment vs Article Volume (Color = Clinical Data %)", fontweight="bold")
ax.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Clinical Data Ratio", fontweight="bold")

# Add firm labels
for firm, row in sentiment_index.iterrows():
    ax.annotate(
        firm.split()[0],  # First word of firm name
        (row["article_count"], row["sentiment_index"]),
        fontsize=8,
        alpha=0.7
    )

# 4. Clinical Data vs Positive Ratio (Bubble chart)
ax = axes[1, 1]
scatter = ax.scatter(
    sentiment_index["clinical_ratio"] * 100,
    sentiment_index["positive_ratio"] * 100,
    s=sentiment_index["article_count"] * 30,
    alpha=0.6,
    c=sentiment_index["sentiment_index"],
    cmap="RdYlGn",
    edgecolors="black",
    linewidth=2
)
ax.set_xlabel("% Articles with Clinical Data", fontweight="bold")
ax.set_ylabel("% Positive Articles", fontweight="bold")
ax.set_title("Clinical Evidence vs Positive Tone (Bubble size = Article Count)", fontweight="bold")
ax.grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Sentiment Index", fontweight="bold")

plt.tight_layout()
plt.show()

print("✓ Visualizations created")

## Section 12: Export Results and Summary

Save the matched dataset and sentiment indices for downstream analysis

In [ ]:
import os

# Create data directories if they don't exist
os.makedirs("data_cache", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("Exporting results...")

# 1. Save matched documents with sentiment features
output_file_matched = "data_cache/glp1_matched_documents.csv"
df_clean[["timestamp", "title", "source", "drugs", "firm", "positive_count", 
          "negative_count", "sentiment_net", "has_clinical_data", "text_length"]].to_csv(
    output_file_matched, index=False
)
print(f"✓ Matched documents: {output_file_matched} ({len(df_clean)} rows)")

# 2. Save sentiment index
output_file_index = "outputs/glp1_sentiment_index.csv"
sentiment_index_export = sentiment_index[
    ["article_count", "sentiment_net_mean", "positive_ratio", "sentiment_ratio",
     "clinical_articles", "clinical_ratio", "content_quality", "sentiment_index"]
].copy()
sentiment_index_export["firm"] = sentiment_index_export.index
sentiment_index_export = sentiment_index_export[
    ["firm", "article_count", "sentiment_net_mean", "positive_ratio", 
     "clinical_articles", "clinical_ratio", "sentiment_index"]
]
sentiment_index_export.to_csv(output_file_index, index=False)
print(f"✓ Sentiment index: {output_file_index} ({len(sentiment_index_export)} firms)")

# 3. Create summary report
summary_report = []
summary_report.append("=" * 100)
summary_report.append("GLP-1 MARKET SENTIMENT ANALYSIS - SUMMARY REPORT")
summary_report.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
summary_report.append("=" * 100)
summary_report.append("")

summary_report.append("DATASET OVERVIEW")
summary_report.append("─" * 100)
summary_report.append(f"Total Articles/Posts: {len(df_original_count)}")
summary_report.append(f"After Drug Matching: {len(df_clean)}")
summary_report.append(f"Average Article Length: {df_clean['word_count'].mean():.0f} words")
summary_report.append(f"Date Range: {df_clean['timestamp'].min().date()} to {df_clean['timestamp'].max().date()}")
summary_report.append("")

summary_report.append("TOP PERFORMING FIRMS (by Sentiment Index)")
summary_report.append("─" * 100)
for rank, (firm, row) in enumerate(sentiment_index.head(5).iterrows(), 1):
    summary_report.append(
        f"{rank}. {firm:30s} | Sentiment: {row['sentiment_index']:6.1f} | "
        f"Articles: {int(row['article_count']):3d} | Clinical Data: {int(row['clinical_articles']):2d}"
    )
summary_report.append("")

summary_report.append("SENTIMENT SOURCES")
summary_report.append("─" * 100)
summary_report.append(f"Articles with Positive Indicators: {(df_clean['positive_count'] > 0).sum()} ({(df_clean['positive_count'] > 0).sum()/len(df_clean)*100:.1f}%)")
summary_report.append(f"Articles with Negative Indicators: {(df_clean['negative_count'] > 0).sum()} ({(df_clean['negative_count'] > 0).sum()/len(df_clean)*100:.1f}%)")
summary_report.append(f"Articles with Clinical Data: {(df_clean['has_clinical_data']).sum()} ({(df_clean['has_clinical_data']).sum()/len(df_clean)*100:.1f}%)")
summary_report.append("")

summary_report.append("KEY INSIGHTS")
summary_report.append("─" * 100)
most_covered = sentiment_index["article_count"].idxmax()
summary_report.append(f"Most Covered Firm: {most_covered} ({int(sentiment_index.loc[most_covered, 'article_count'])} mentions)")

highest_sentiment = sentiment_index["sentiment_index"].idxmax()
summary_report.append(f"Highest Sentiment: {highest_sentiment} (Index: {sentiment_index.loc[highest_sentiment, 'sentiment_index']:.1f})")

most_clinical = sentiment_index["clinical_ratio"].idxmax()
summary_report.append(f"Most Research-Driven: {most_clinical} ({sentiment_index.loc[most_clinical, 'clinical_ratio']*100:.1f}% clinical articles)")

summary_report.append("")
summary_report.append("=" * 100)

# Write report
report_file = "outputs/sentiment_analysis_report.txt"
with open(report_file, "w") as f:
    f.write("\n".join(summary_report))

print(f"✓ Summary report: {report_file}")

# Print summary to console
print("\n" + "\n".join(summary_report))

## Section 13: Advanced Analysis - Firm Comparison

Compare firms across multiple dimensions

In [ ]:
# Create comparison matrix
print("\nADVANCED ANALYSIS: FIRM POSITIONING MATRIX")
print("=" * 120)

# Categorize firms by strategy
market_leaders = ["Novo Nordisk A/S", "Eli Lilly and Company"]
emerging_challengers = ["Amgen Inc.", "Roche Holding AG", "Viking Therapeutics, Inc."]
specialists = ["Pfizer Inc.", "AstraZeneca Plc", "Structure Therapeutics, Inc.", "Zealand Pharma A/S", "Boehringer Ingelheim GmbH"]

categories = {
    "Market Leaders": market_leaders,
    "Emerging Challengers": emerging_challengers,
    "Specialists": specialists,
}

for category, firms in categories.items():
    print(f"\n{category}")
    print("─" * 120)
    
    category_firms = sentiment_index[sentiment_index.index.isin(firms)]
    category_mean_sentiment = category_firms["sentiment_index"].mean()
    category_total_articles = category_firms["article_count"].sum()
    
    print(f"Average Sentiment Index: {category_mean_sentiment:.1f}")
    print(f"Total Article Mentions: {int(category_total_articles)}")
    print(f"Average Clinical Data Ratio: {category_firms['clinical_ratio'].mean()*100:.1f}%")
    print()
    
    for firm in firms:
        if firm in sentiment_index.index:
            row = sentiment_index.loc[firm]
            print(f"  • {firm:35s} | Sentiment: {row['sentiment_index']:6.1f} | "
                  f"Articles: {int(row['article_count']):3d} | Clinical %: {row['clinical_ratio']*100:5.1f}%")

# Drug strength analysis
print("\n" + "=" * 120)
print("DRUG PORTFOLIO ANALYSIS")
print("─" * 120)

drug_analysis = df_clean.groupby("drugs").agg({
    "text": "count",
    "sentiment_net": "mean",
    "has_clinical_data": "mean",
    "firm": lambda x: x.iloc[0] if len(x) > 0 else "Unknown"
}).rename(columns={"text": "count"})

drug_analysis = drug_analysis.sort_values("count", ascending=False)

for drug, row in drug_analysis.iterrows():
    print(f"  {drug:20s} | Mentions: {int(row['count']):3d} | "
          f"Avg Sentiment: {row['sentiment_net']:5.2f} | Clinical Data: {row['has_clinical_data']*100:5.1f}% | "
          f"Manufacturer: {row['firm']}")

## Section 14: Usage Guide and Next Steps

Instructions for using this analysis for downstream tasks

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════════════╗
║                        ANALYSIS COMPLETE - NEXT STEPS                          ║
╚════════════════════════════════════════════════════════════════════════════════╝

OUTPUT FILES CREATED:
────────────────────────────────────────────────────────────────────────────────

1. data_cache/glp1_matched_documents.csv
   - All matched documents with drug and firm assignments
   - Includes sentiment features (positive/negative indicators, clinical data flags)
   - 605 rows × 10 columns

2. outputs/glp1_sentiment_index.csv
   - Firm-level sentiment indices
   - Key metrics: article count, avg sentiment, clinical data ratio
   - Ready for econometric analysis

3. outputs/sentiment_analysis_report.txt
   - Text summary of key findings
   - Market positioning overview

DOWNSTREAM APPLICATIONS:
────────────────────────────────────────────────────────────────────────────────

A. SENTIMENT TIME SERIES:
   - Aggregate sentiment by firm and time period (daily/weekly/monthly)
   - Detect turning points and sentiment shocks
   - Use for event study analysis

B. CORRELATION WITH STOCK RETURNS:
   - Merge sentiment index with firm stock prices
   - Calculate rolling correlation
   - Run Granger causality tests

C. DIFFERENCE-IN-DIFFERENCES ANALYSIS:
   - Compare treated firms (announced clinical trials) vs control
   - Measure sentiment response to news events
   - Quantify market impact

D. PANEL DATA MODELING:
   - Fixed effects: firm-specific sentiment patterns
   - Time effects: market-wide GLP-1 sentiment cycles
   - Random effects: between-firm heterogeneity

E. MACHINE LEARNING:
   - Use sentiment index features for portfolio optimization
   - Predict stock volatility from sentiment changes
   - Classify documents with advanced NLP models (FinBERT, etc.)

KEY METRICS AVAILABLE:
────────────────────────────────────────────────────────────────────────────────

• article_count: Media coverage volume (proxy for attention)
• sentiment_net_mean: Average net sentiment tone
• positive_ratio: Proportion of positive-tone documents
• clinical_ratio: Articles mentioning clinical trials/data
• sentiment_index: Composite 0-100 index
• content_quality: Articles with substantive content (trial data, etc.)

EXAMPLE CODE:
────────────────────────────────────────────────────────────────────────────────

# Load results
sentiment_idx = pd.read_csv('outputs/glp1_sentiment_index.csv', index_col='firm')
matched_docs = pd.read_csv('data_cache/glp1_matched_documents.csv')

# Time series by firm
ts = matched_docs.groupby(['timestamp', 'firm']).agg({
    'sentiment_net': 'mean',
    'text': 'count'
}).reset_index()

# Merge with stock returns (assuming returns_df available)
analysis_df = ts.merge(returns_df, on=['timestamp', 'firm'])

# Run panel regression
from linearmodels.panel import RandomEffects
re_model = RandomEffects(analysis_df.set_index(['firm', 'timestamp']).astype(float))
results = re_model.fit()
print(results)

════════════════════════════════════════════════════════════════════════════════
""")